# Assignment 04 – Your First Model
**Domain:** Cybersecurity – Clinic Security Monitoring  
**Dataset:** UCI PhiUSIIL Phishing URL dataset.

**Prediction task:** Classification. Predict whether a URL is phishing (positive class 1) or legitimate (class 0). UCI's original labels are 0 = phishing and 1 = legitimate; this notebook explicitly recodes the target so phishing is the positive class.

## 1. Load data, choose target and features
Use numeric URL-structure features and exclude identifiers, raw text fields, and target-derived similarity/probability fields where possible. This is a baseline exercise, not a production security detector.

In [ ]:
# Load the official UCI PhiUSIIL phishing URL dataset.
# UCI label convention: 1 = legitimate, 0 = phishing.
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=967)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = [str(c).strip() for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label column candidates:", [c for c in df.columns if c.lower() == "label"])
display(df.head())


label_col = next(c for c in df.columns if c.lower() == "label")
# UCI convention: 0 = phishing, 1 = legitimate. Recode phishing as positive class (1).
y_all = (pd.to_numeric(df[label_col], errors="coerce") == 0).astype(int)

# Use numeric URL-structure features only; exclude identifiers and target-derived similarity/probability fields.
preferred = [
    "URLLength", "DomainLength", "IsDomainIP", "NoOfSubDomain",
    "NoOfObfuscatedChar", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL",
    "NoOfQMarkInURL", "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "IsHTTPS", "NoOfURLRedirect", "NoOfSelfRedirect"
]
available = {c.lower(): c for c in df.columns}
feature_cols = [available[c.lower()] for c in preferred if c.lower() in available]
if len(feature_cols) < 3:
    excluded = {"label", "filename", "url", "domain", "tld", "title",
                "urlsimilarityindex", "urltitlematchscore", "urlcharprob",
                "tldlegitimateprob"}
    feature_cols = [
        c for c in df.select_dtypes(include=np.number).columns
        if c.lower() not in excluded and c.lower() != label_col.lower()
    ][:10]

X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce")
print("Target: phishing (1) vs legitimate (0)")
print("Features:", feature_cols)
print("Class balance:")
display(y_all.value_counts(normalize=True).rename("proportion").to_frame())

## 2. Train/test split
Hold back 20% for testing and use a fixed random state for reproducibility. Stratification preserves the class proportions.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

# Use a stratified sample for fast, repeatable Colab experiments while retaining the full dataset for exploration.
sample_size = min(20000, len(X_all))
sample_index = y_all.groupby(y_all).sample(
    frac=1, random_state=42
).index[:sample_size]
# Rebuild a stratified sample using group-wise sampling
sample_parts = []
for cls, group in y_all.groupby(y_all):
    n_cls = max(1, round(sample_size * len(group) / len(y_all)))
    sample_parts.append(group.sample(n=min(n_cls, len(group)), random_state=42).index)
sample_index = pd.Index(np.concatenate(sample_parts))
X = X_all.loc[sample_index]
y = y_all.loc[sample_index]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 3. Train a baseline logistic regression model
The preprocessing steps are kept inside a pipeline. Score the model on both training and test sets.

In [ ]:
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
baseline_model.fit(X_train, y_train)

train_score = baseline_model.score(X_train, y_train)
test_score = baseline_model.score(X_test, y_test)
print("Logistic regression training accuracy:", round(train_score, 4))
print("Logistic regression test accuracy:", round(test_score, 4))
print("Train-test gap:", round(train_score - test_score, 4))

## 4. Explain the train/test gap
A training score measures performance on examples used to fit the model; a test score measures performance on held-out examples. A much higher training score indicates possible overfitting. Similar scores suggest better generalisation, although performance must also be assessed using precision and recall.

In [ ]:
gap = train_score - test_score
if gap > 0.10:
    print("The training score is substantially higher; this suggests overfitting.")
else:
    print("The train-test gap is relatively small; inspect other metrics before concluding the model generalises well.")

## 5. Deliberately overfit a decision tree
An unrestricted decision tree can memorise training rows. Record both scores to demonstrate the effect.

In [ ]:
overfit_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(random_state=42))
])
overfit_tree.fit(X_train, y_train)
tree_train = overfit_tree.score(X_train, y_train)
tree_test = overfit_tree.score(X_test, y_test)
print("Unrestricted tree training accuracy:", round(tree_train, 4))
print("Unrestricted tree test accuracy:", round(tree_test, 4))
print("Unrestricted tree train-test gap:", round(tree_train - tree_test, 4))

## 6. Compute a simple majority-class baseline
For classification, the appropriate simple baseline is a classifier that always predicts the most common class. The trained model should be compared with this baseline on the same test set.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
baseline_accuracy = dummy.score(X_test, y_test)
print("Majority-class baseline accuracy:", round(baseline_accuracy, 4))
print("Logistic regression test accuracy:", round(test_score, 4))
print("Model beats majority baseline:", test_score > baseline_accuracy)

## 7. Interpretation and limitations
The target is phishing versus legitimate URL classification. The chosen features are URL-structure indicators, and the raw URL, filename and target-derived similarity fields are excluded from this baseline. The fixed 80/20 split makes the experiment reproducible. A training score much higher than the test score is evidence of overfitting; the unrestricted tree is included to make this risk visible. The majority-class DummyClassifier provides a minimum benchmark. This random split is a teaching baseline and may still overestimate performance on entirely new domains; a domain-grouped evaluation would be stronger.

## Completion checklist
- [ ] Target chosen and task correctly identified as classification.
- [ ] Features listed and leakage-prone identifiers/derived fields excluded deliberately.
- [ ] 80/20 split uses fixed `random_state=42`.
- [ ] Model scored on training and test sets.
- [ ] Train/test gap explained.
- [ ] Unrestricted decision tree trained and both scores recorded.
- [ ] Majority-class baseline computed and compared; report honestly whether the model beats it.